# Named Entity Recognition and Classification
## Install/import libraries

In [1]:
from collections import Counter 
import pandas as pd
from sklearn.metrics import classification_report
from sklearn import svm
import gensim
import csv
from nltk.corpus.reader import ConllCorpusReader

In [2]:
# Load the IPM NEL dataset
path_to_train_folder = "training_sets/"
conll_path = f"{path_to_train_folder}ipm_nel_corpus/"

In [33]:
import nltk
from nltk.corpus import stopwords

# Lists to store tokens and labels
tokens_list = []
labels_list = []

stop_words = set(stopwords.words('english'))

with open(f"{conll_path}ipm_nel.conll", 'r', encoding='utf-8') as f:
    for line in f:
        if not line:  # skip empty lines
            continue

        # Split line by whitespace (or use '\t' if the file is tab-separated)
        parts = line.split(sep='\t')
        
        # The first field is the token and the third is the entity label.
        token = parts[0]
        label = parts[2].upper()

        # Skip null values, usernames and urls
        if not token or not label or token[0] == '@' or token[:4] == 'http' or token[0] == "#" or token.lower() in stop_words:
            continue
        
        
        tokens_list.append(token)
        labels_list.append(label)

# Create the DataFrame
training_ner = pd.DataFrame({
    'token': tokens_list,
    'BIO_NER_tag': labels_list
})

print(training_ner.head(20))


          token BIO_NER_tag
0        lineup           O
1       tonight           O
2             .           O
3     Keppinger    B-PERSON
4          sits           O
5             ,           O
6         Downs    B-PERSON
7         plays           O
8            2B           O
9             ,           O
10           CJ    B-PERSON
11         bats           O
12          5th           O
13            .           O
14      staying           O
15  Gainesville   B-GEO-LOC
16            ,           O
17         come           O
18        check           O
19        Costa  B-FACILITY


In [34]:
# counting frequency of the NER tags that show up in the training and test data
testing_ner = pd.read_csv("test_sets/NER-test.tsv", sep="\t", on_bad_lines='skip', header=None, skiprows=1)
testing_ner.columns = ['token_id', 'sentence_id', 'token', 'BIO_NER_tag']

ner_labels_train = training_ner['BIO_NER_tag']
ner_labels_test = testing_ner['BIO_NER_tag']

In [46]:
mapping = {
    'O': 'O', 
    'MISC': 'O', 
    'B-OTHER': 'B-ORG', 
    'I-OTHER': 'I-ORG', 
    'B-PRODUCT': 'B-ORG', 
    'I-PRODUCT': 'I-ORG', 
    'PERSON': 'B-PERSON', 
    'B-PERSON': 'B-PERSON', 
    'I-PERSON': 'I-PERSON', 
    'B-MUSICARTIST': 'B-PERSON', 
    'I-MUSICARTIST': 'I-PERSON', 
    'B-TVSHOW': 'B-WORK_OF_ART', 
    'I-TVSHOW': 'I-WORK_OF_ART', 
    'B-MOVIE': 'B-WORK_OF_ART', 
    'I-MOVIE': 'I-WORK_OF_ART', 
    'LOCATION': 'B-LOC', 
    'B-GEO-LOC': 'B-LOC', 
    'I-GEO-LOC': 'I-LOC', 
    'ORGANIZATION': 'B-ORG', 
    'B-COMPANY': 'B-ORG', 
    'I-COMPANY': 'I-ORG', 
    'B-FACILITY': 'B-LOC', 
    'I-FACILITY': 'I-LOC', 
    'B-SPORTSTEAM': 'B-ORG', 
    'I-SPORTSTEAM': 'I-ORG', 
}
print("before", training_ner[:10])
training_ner['BIO_NER_tag'] = training_ner['BIO_NER_tag'].map(mapping)

training_gold_labels = training_ner['BIO_NER_tag'].to_numpy()
training_features = training_ner['token'].to_numpy()

test_gold_labels = testing_ner['BIO_NER_tag'].to_numpy()
test_features = testing_ner['token'].to_numpy()

print("after",training_ner[50:60])

before        token BIO_NER_tag
0     lineup           O
1    tonight           O
2          .           O
3  Keppinger    B-PERSON
4       sits           O
5          ,           O
6      Downs    B-PERSON
7      plays           O
8         2B           O
9          ,           O
after          token BIO_NER_tag
50      friend           O
51        camp           O
52           .           O
53     camping           O
54  Robinhoods         NaN
55         bay         NaN
56      Jasmin         NaN
57           .           O
58        Good           O
59     weekend           O


Analyzing training and test data (im)balance:

In [87]:
train_total = sum(train_label_counts.values())
test_total = sum(test_label_counts.values())

print("\nPercentage distribution in training data:")
for label, count in train_label_counts.most_common():
    percentage = (count / train_total) * 100
    print(f"{label}: {percentage}%")

print("\nPercentage distribution in test data:")
for label, count in test_label_counts.most_common():
    percentage = (count / test_total) * 100
    print(f"{label}: {percentage}%")


Percentage distribution in training data:
O: 86.47171358294617%
B-ORG: 3.006285870456409%
I-ORG: 2.5690079256627496%
B-PERSON: 2.323039081716316%
B-LOC: 2.213719595517901%
I-PERSON: 1.8037715222738453%
I-LOC: 0.8198961464881116%
B-WORK_OF_ART: 0.43727794479365945%
I-WORK_OF_ART: 0.3552883301448483%

Percentage distribution in test data:
O: 78.38983050847457%
B-PERSON: 4.661016949152542%
I-WORK_OF_ART: 4.23728813559322%
B-WORK_OF_ART: 3.8135593220338984%
I-PERSON: 3.389830508474576%
B-LOC: 2.9661016949152543%
B-ORG: 1.2711864406779663%
I-ORG: 0.847457627118644%
I-LOC: 0.423728813559322%


In [36]:
# Load word embedding model
gensim_path = 'GoogleNews-vectors-negative300.bin\\GoogleNews-vectors-negative300.bin'
word_embedding_model = gensim.models.KeyedVectors.load_word2vec_format(gensim_path, binary=True)

# Initialize arrays
training_vec = []
test_vec = []

# Obtain vectors for training features
for word in training_features:
    # Is word in the model vocabulary (loaded with the Google word2vec embeddings)?
    if word in word_embedding_model:
        vector = word_embedding_model[word]  # assign embedding vector value to variable
    else: 
        vector = [0] * 300  # Create a vector with 300 zeros, as word2vec model has 300 dimensions

    training_vec.append(vector)

# Obtain vectors for test features
for word in test_features:
    # Is word in the model vocabulary (loaded with the Google word2vec embeddings)?
    if word in word_embedding_model:
        vector = word_embedding_model[word]  # assign embedding vector value to variable
    else: 
        vector = [0] * 300  # Create a vector with 300 zeros as word2vec model has 300 dimensions

    test_vec.append(vector)

FileNotFoundError: [Errno 2] No such file or directory: 'GoogleNews-vectors-negative300.bin\\GoogleNews-vectors-negative300.bin'

NOTE! Run either the vectorization above or the one below

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vec = TfidfVectorizer()
concatenated_features = np.concatenate((training_features, test_features), axis=0)
concat_vec = vec.fit_transform(concatenated_features)
split_idx = len(training_features)
training_vec, test_vec = concat_vec[:split_idx], concat_vec[split_idx:]

## Training the Support Vector Machine
Since the training data contains a varying amount of instances across domains, there is a need for a model that is robust against overfitting. SVMs provide robustness via the use of a regularization parameter, often denoted as `C`. Furthermore, SVMs are based on the maximization of margins between classes. This often leads to good generalization performance, which is crucial for our task of minimizing misclassifications (especially across similar entity types). 

We choose the linear kernel, as the linear kernel of SVMs may outperform the polynomial and RBF kernel for the NER task (Alokaili and Menai, 2019).

In [38]:
lin_clf = svm.LinearSVC()
#from sklearn.svm import LinearSVC
#lin_clf = LinearSVC(class_weight='balanced')

# NOTE: DON'T run this cell again if you've already trained the SVM! It may take up to 10 minutes!
lin_clf.fit(training_vec, training_gold_labels)

LinearSVC()

## Make predictions for the test set and evaluate
Make predictions with the model (predict the NER labels of the tokens in the test set).

In [39]:
# Obtain predictions from the SVM model
svm_predictions = lin_clf.predict(test_vec)

# Generate the classification report and print it
class_report = classification_report(test_gold_labels, svm_predictions)
print(class_report)

               precision    recall  f1-score   support

        B-LOC       0.00      0.00      0.00         7
        B-ORG       0.00      0.00      0.00         3
     B-PERSON       1.00      0.09      0.17        11
B-WORK_OF_ART       0.00      0.00      0.00         9
        I-LOC       0.00      0.00      0.00         1
        I-ORG       0.00      0.00      0.00         2
     I-PERSON       1.00      0.12      0.22         8
I-WORK_OF_ART       0.00      0.00      0.00        10
            O       0.80      0.99      0.89       185

     accuracy                           0.78       236
    macro avg       0.31      0.13      0.14       236
 weighted avg       0.71      0.78      0.71       236



/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#TODO: Analyze classification report

* Which NERC labels does the classifier perform well on? Why do you think this is the case?
* Which NERC labels does the classifier perform poorly on? Why do you think this is the case?
